# 02 Data cleaning and target

Cleaned-up version of the EDA from Oct 16–17. Every section states its purpose; the
decisions and evidence are recorded in `DECISIONS.md`. Shared functions live in `src/target.py`.

Run top to bottom. Data files go in `../data/` (not tracked by git).

## 0. Setup

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from src.target import (DATA_DIR, LINES, YEARS, VALUATION_DATE, LARGE_LOSS_CAP, DUP_KEY,
                        load_policies, load_claims, match_claims, build_target,
                        cost_per_exposure, completeness)

pol  = {l: load_policies(l, "train") for l in LINES}
test = {l: load_policies(l, "test") for l in LINES}
raw    = load_claims(drop_duplicates=False, drop_repudiated=False)   # untouched, for checks
claims = load_claims()                                               # cleaned, used for the target
print({l: len(pol[l]) for l in LINES}, "| claims raw:", len(raw), "| cleaned:", len(claims))

## 1. Data structure
**Purpose:** confirm the unit of analysis, the train/test periods, how policy periods connect,
and whether test policies have history in train.

In [ ]:
m, tm = pol["motor"], test["motor"]

# row_id is the unit (one policy period); policy_id repeats across renewals and mid-term changes
print("policy_id unique in train motor:", m.policy_id.is_unique, "| row_id unique:", m.row_id.is_unique)
print("columns only in train:", set(m.columns) - set(tm.columns))

# Train/test periods and the data valuation date
print("train:", m.period_start.min().date(), "~", m.period_start.max().date())
print("test :", tm.period_start.min().date(), "~", tm.period_start.max().date())
print("last loss_date:", raw.loss_date.max().date(), "| last report_date:", raw.report_date.max().date())

# Consecutive periods of a policy: gap between period_end and the next period_start (days)
s = m.sort_values(["policy_id", "period_start"])
gap = (s.groupby("policy_id").period_start.shift(-1) - s.period_end).dt.days
print("\ngap (days):\n", gap.value_counts().head())

# Rows spanning two calendar years
print("\nrows spanning two years:", {l: int((pol[l].period_start.dt.year != pol[l].period_end.dt.year).sum()) for l in LINES})

# Share of test policies with earlier periods in train (history features)
print("test policies seen in train:", round(tm.policy_id.isin(m.policy_id).mean(), 4))

## 2. Exposure correction
**Purpose:** find rows whose exposure disagrees with the period dates (360-day basis, or
recorded in days) and measure how much the correction changes total exposure.

In [ ]:
for l in LINES:
    print(f"\n[train {l}]", pol[l].exp_flag.value_counts().to_dict(),
          "| [test]", test[l].exp_flag.value_counts().to_dict())
    by_year = pol[l].groupby("year")[["exposure", "exposure_clean"]].sum()
    by_year["change_%"] = (by_year.exposure_clean / by_year.exposure - 1) * 100
    print(by_year.round(1))

## 3. Claims cleaning
**Purpose:** check claim statuses and amounts, then confirm the duplicate records
(identical except `claim_id`) and repudiated claims removed by `load_claims()`.

In [ ]:
# Status by line: closed claims have paid == incurred; open claims carry case reserves
print(raw.groupby(["line", "status"])[["paid_to_date", "incurred_to_date", "recoveries"]]
         .agg(["count", "sum", "max"]).round(0))

In [ ]:
# Duplicates: same policy, peril, dates, status and amounts, different claim_id
print("duplicated claim_id:", raw.claim_id.duplicated().sum())
dup = raw.duplicated(subset=DUP_KEY, keep="first")
print("\nduplicate records by line:")
print(raw[dup].groupby("line").agg(n=("claim_id", "size"),
                                   incurred_m=("incurred_to_date", lambda s: s.sum() / 1e6)).round(2))
print("\nby loss year:")
print(raw[dup].groupby(["line", "loss_year"]).size().unstack(fill_value=0))

# Same amount but different report date or status (ambiguous cases)
amb = raw.duplicated(subset=["policy_id", "line", "peril", "loss_date", "incurred_to_date"],
                     keep="first") & ~dup
print("\nambiguous near-duplicates:", int(amb.sum()))

# Repudiated claims are all zero
rep = raw[raw.status == "repudiated"]
print("\nrepudiated:", len(rep), "| max amount:", rep[["paid_to_date", "incurred_to_date", "recoveries"]].max().max())

## 4. Matching claims to policy periods
**Purpose:** link each claim to the period containing its loss date, and check the match
rate, unmatched claims, and claims matching more than one period.

In [ ]:
for l in LINES:
    c = claims[claims.line == l].merge(
        pol[l][["row_id", "policy_id", "period_start", "period_end"]], on="policy_id", how="left")
    c["in_period"] = (c.loss_date >= c.period_start) & (c.loss_date <= c.period_end)
    hit = c[c.in_period]
    n_all = claims.loc[claims.line == l, "claim_id"].nunique()
    n_hit = hit.claim_id.nunique()
    multi = int((hit.groupby("claim_id").size() > 1).sum())
    unmatched = claims[(claims.line == l) & ~claims.claim_id.isin(hit.claim_id)]
    print(f"{l}: {n_hit}/{n_all} matched ({n_hit / n_all:.2%}), multi-period {multi}, "
          f"unmatched by loss year {unmatched.loss_year.value_counts().sort_index().to_dict()}")

In [ ]:
# Several claims on the same row and loss date: is excess per claim or per event an issue?
mc_all = match_claims(claims, pol["motor"], "motor")
multi = mc_all.groupby(["row_id", "loss_date"]).size()
print(multi.value_counts().sort_index())
ev = mc_all.set_index(["row_id", "loss_date"]).loc[multi[multi > 1].index]
print(ev.groupby(level=[0, 1]).peril.apply(lambda s: " + ".join(sorted(s))).value_counts().head(10))

## 5. Target definition
**Purpose:** support treating `incurred_to_date` as gross of excess, measure the effect of not
deducting excess on liability perils, and verify `build_target` against a manual calculation.

In [ ]:
# Closed motor claims at or below the policy excess (ground-up amounts are recorded)
c = match_claims(claims, pol["motor"], "motor", excess_on_liability=True, cap=None)
closed = c[c.status == "closed"]
print(closed.groupby("peril").agg(
    n=("claim_id", "size"),
    share_le_excess=("incurred_to_date", lambda s: (s <= closed.loc[s.index, "excess"]).mean()),
    median_incurred=("incurred_to_date", "median"),
    median_excess=("excess", "median")).round(3))

In [ ]:
# Excess on liability perils: both versions, uncapped
both = pd.DataFrame({
    "excess_all": match_claims(claims, pol["motor"], "motor", True, None).groupby("peril").net.sum(),
    "no_excess_on_liability": match_claims(claims, pol["motor"], "motor", False, None).groupby("peril").net.sum(),
}) / 1e6
both.loc["total"] = both.sum()
print(both.round(2))

In [ ]:
# Manual check: rows with 2+ claims, recomputed by hand
for l in LINES:
    c = match_claims(claims, pol[l], l, cap=None)
    tgt = pd.Series(build_target(claims, pol[l], l), index=pol[l].row_id)
    for rid in c.row_id.value_counts().loc[lambda s: s >= 2].index[:2]:
        rows = c[c.row_id == rid]
        manual = sum(max(max(r.incurred_to_date - r.excess, 0) - r.recoveries, 0) for r in rows.itertuples())
        print(f"{l} {rid}: manual {manual:,.2f} | build_target {tgt[rid]:,.2f}")

## 6. Net cost per exposure by year
**Purpose:** first view of level and trend by line, using corrected exposure and cleaned claims (uncapped).

In [ ]:
cpe = pd.DataFrame({l: cost_per_exposure(pol[l], build_target(claims, pol[l], l)) for l in LINES})
print(cpe.round(1))

## 7. Large losses
**Purpose:** choose the cap for individual motor claims by looking at the bodily injury
distribution and how much cost each cap removes.

In [ ]:
mc = match_claims(claims, pol["motor"], "motor")          # excess not on liability, cap 500k
bi = mc[mc.peril == "bodily_injury"]
print(bi.incurred_to_date.quantile([0.5, 0.9, 0.99, 0.999]).round(0))
total = bi.net.sum()
for cap in [250_000, 500_000, 1_000_000, 2_000_000]:
    print(f"cap {cap:>9,}: {(bi.net > cap).sum():>3} claims, {np.maximum(bi.net - cap, 0).sum() / total:.1%} of BI cost above cap")

print(mc.nlargest(5, "incurred_to_date")[["claim_id", "peril", "status", "loss_date", "paid_to_date", "incurred_to_date"]])
print(bi.assign(above=np.maximum(bi.net - 1_000_000, 0)).groupby("loss_year")[["net", "above"]].sum().div(1e6).round(2))

## 8. Development: reporting-delay completeness
**Purpose:** estimate the share of each loss year reported by the valuation date (from 2021
report lags) and gross up reported cost to ultimate. Capped at 500k; the excess is a loading.

In [ ]:
exp_by_year = pol["motor"].groupby("year").exposure_clean.sum()
dev = pd.DataFrame([
    {"peril": p, "year": y,
     "reported_m": mc.loc[(mc.peril == p) & (mc.loss_year == y), "net_cap"].sum() / 1e6,
     "completeness": completeness(mc, p, y)}
    for p in mc.peril.unique() for y in YEARS])
dev["ultimate_m"] = dev.reported_m / dev.completeness
print(dev.round(3).to_string(index=False))

summary = dev.groupby("year")[["reported_m", "ultimate_m"]].sum()
summary["reported_per_exp"] = summary.reported_m * 1e6 / exp_by_year
summary["ultimate_per_exp"] = summary.ultimate_m * 1e6 / exp_by_year
print(summary.round(1))
print("large-loss loading per exposure:", round(np.maximum(mc.net - LARGE_LOSS_CAP, 0).sum() / exp_by_year.sum(), 1))

In [ ]:
# Are late-reported BI claims larger? Count- vs amount-based completeness
ref = bi[bi.loss_year == 2021].copy()
ref["lag_bucket"] = pd.cut(ref.report_lag, [-1, 30, 90, 180, 365, 730, 5000],
                           labels=["~1m", "1-3m", "3-6m", "6-12m", "1-2y", "2y+"])
print(ref.groupby("lag_bucket", observed=True).agg(n=("claim_id", "size"), avg_net=("net_cap", "mean")).round(0))

print(pd.DataFrame([{"year": y,
                     "compl_count": completeness(mc, "bodily_injury", y),
                     "compl_amount": completeness(mc, "bodily_injury", y, weight_col="net_cap")}
                    for y in YEARS]).round(3).to_string(index=False))

## 9. Bodily injury: 2023 severity change
**Purpose:** BI stays ~20% low after the completeness adjustment. Split into frequency and
severity, and test whether the 2023 severity drop is a real step change.

In [ ]:
g = bi.groupby("loss_year").agg(n=("claim_id", "size"), net_m=("net_cap", lambda s: s.sum() / 1e6),
                                open_share=("status", lambda s: (s == "open").mean()))
g["compl"] = [completeness(mc, "bodily_injury", y) for y in g.index]
g["freq_per_1000"] = g.n / g.compl / exp_by_year * 1000
g["avg_sev"] = g.net_m * 1e6 / g.n
print(g.round(3))
print(bi.groupby(["loss_year", "status"]).net_cap.mean().unstack().round(0))

In [ ]:
# Step or gradual? Quarterly severity; like-for-like (reported within 365 days); other perils
print(bi.groupby(bi.loss_date.dt.to_period("Q")).net_cap.agg(["size", "mean", "median"]).round(0).to_string())
print(bi[bi.report_lag <= 365].groupby("loss_year").net_cap.agg(["size", "mean", "median"]).round(0))
print(mc.groupby(["peril", "loss_year"]).net_cap.median().unstack().round(0))

## 10. Motor frequency and severity trends
**Purpose:** set trends for projection to 2025. Frequency is completeness-adjusted; severity is
gross of excess (capped) to avoid the deductible leverage effect. BI change is post-reform only.

In [ ]:
fs = pd.DataFrame([
    {"peril": p, "year": y,
     "freq_per_1000": len(s) / completeness(mc, p, y) / exp_by_year[y] * 1000,
     "sev_gross": np.minimum(s.incurred_to_date, LARGE_LOSS_CAP).mean()}
    for p in mc.peril.unique() for y in YEARS
    for s in [mc[(mc.peril == p) & (mc.loss_year == y)]]])

def annual_change(t):
    out = (t[2024] / t[2021]) ** (1 / 3) - 1
    out["bodily_injury"] = t.loc["bodily_injury", 2024] / t.loc["bodily_injury", 2023] - 1
    return out

for col in ["freq_per_1000", "sev_gross"]:
    t = fs.pivot(index="peril", columns="year", values=col)
    t["annual_%"] = annual_change(t) * 100
    print(f"\n[{col}]\n", t.round(2))

## Next
- Check whether the 2023 drop in own_damage / third_party_prop frequency is portfolio mix
  (driver_assist, telematics, new business, cover type).
- Build the modelling target: on-level 2021–22 BI (×0.83), trend severity to 2025, cap 500k.
- Apply completeness adjustment to buildings and contents; split buildings trend into frequency and severity.
- Check `customer_service_contacts` for leakage and `weather` for 2025 values.